# 1. PDF 로드 및 청크 분할

## 1) 환경 설정 및 라이브러리 임포트

In [18]:
import os
from dotenv import load_dotenv

# LangChain 문서 로더 및 분할기
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 현재 경로 기준으로 .env 파일 로드
load_dotenv()

# 사용할 데이터 폴더 경로 설정
DATA_PATH = "data/"

# 로드할 파일 목록
FILE_SOURCES = {
    "ai_rmf": "nist_ai_risk_framework.pdf",
    "csf_2_0": "nist_cybersecurity_framework.pdf",
    "zero_trust": "nist_zero_trust.pdf"
}

print(f"데이터 경로: {DATA_PATH}")

데이터 경로: data/


## 2) PDF 파일 로드 및 메타데이터 추가

In [19]:
# 전체 문서를 저장할 리스트
all_documents = []

for source_key, filename in FILE_SOURCES.items():
    file_path = os.path.join(DATA_PATH, filename)
    
    # 1. LangChain PDF Loader로 문서 로드
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    
    # 2. 메타데이터 설정
    # 각 document 객체에 지정된 메타데이터를 추가
    for doc in documents:
        # 파일 이름에서 확장자 제거 후 doc_title 설정 (예시)
        doc_title = filename.split('(')[0].strip()
        
        doc.metadata["source"] = source_key
        doc.metadata["page"] = doc.metadata.pop("page")  # 기존 'page'를 그대로 사용하거나 재정의
        doc.metadata["doc_title"] = doc_title
    
    all_documents.extend(documents)
    print(f"✅ {source_key} 문서 로드 완료. 총 {len(documents)} 페이지.")

print(f"\n총 로드된 페이지/문서 수: {len(all_documents)}")

✅ ai_rmf 문서 로드 완료. 총 48 페이지.
✅ csf_2_0 문서 로드 완료. 총 32 페이지.
✅ zero_trust 문서 로드 완료. 총 59 페이지.

총 로드된 페이지/문서 수: 139


## 3) 청크 분할

In [20]:
# RecursiveCharacterTextSplitter를 사용하여 청크 분할
text_splitter = RecursiveCharacterTextSplitter(
    # 요구사항: 700~1000자 권장
    chunk_size=900, 
    chunk_overlap=150, # 적절한 overlap 설정
    length_function=len,
    is_separator_regex=False,
)

# 텍스트 청크 분할 실행
chunks = text_splitter.split_documents(all_documents)

print(f"✅ 원본 문서 총 페이지 수: {len(all_documents)}")
print(f"✅ 분할된 최종 청크 수: {len(chunks)}")
print("-" * 30)
print(f"첫 번째 청크 예시:\n")
print(chunks[0].page_content[:200] + "...")
print(f"메타데이터: {chunks[0].metadata}")

✅ 원본 문서 총 페이지 수: 139
✅ 분할된 최종 청크 수: 484
------------------------------
첫 번째 청크 예시:

NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)...
메타데이터: {'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-01-24T14:45:46-05:00', 'author': 'National Institute of Standards and Technology', 'keywords': 'Artificial Intelligence (AI); AI; AI RMF; AI RMF 1.0; AI systems; trustworthy and responsible AI.', 'moddate': '2025-06-04T13:01:45-04:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': 'As directed by the National Artificial Intelligence Initiative Act of 2020 (P.L. 116-283), the goal of the AI RMF is to offer a resource to the organizations designing, developing, deploying, or using AI systems to help manage the many risks of AI and promote trustworthy and responsible development and use of AI systems. The Framework is intended to be voluntary, rights-preserving, 

# 2. Pinecone 인덱스 생성 및 청크 업서트

In [21]:
# --- 라이브러리 임포트 ---
import os
from langchain_openai import OpenAIEmbeddings # OpenAI 임베딩 모델 사용
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# --- 설정 변수 ---
INDEX_NAME = "nist-rag-index"
EMBEDDING_MODEL = "text-embedding-3-small"

In [5]:
# 1. 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=os.getenv("OPENAI_API_KEY")
)
print(f"✅ 임베딩 모델 초기화 완료: {EMBEDDING_MODEL}")

# 2. Pinecone 클라이언트 초기화
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
print("✅ Pinecone 클라이언트 초기화 완료.")

✅ 임베딩 모델 초기화 완료: text-embedding-3-small
✅ Pinecone 클라이언트 초기화 완료.


In [24]:
# 인덱스 목록 가져오기
index_list = pc.list_indexes()

# 인덱스 이름만 추출
index_names = [idx["name"] for idx in index_list]

if INDEX_NAME not in index_names:
    print(f"⏳ 인덱스 '{INDEX_NAME}'가 존재하지 않아 새로 생성합니다...")
    
    # Serverless 환경을 위한 Spec 정의 (AWS, us-west-2는 예시)
    # 실제 환경에 맞게 region을 변경해야 할 수 있습니다.
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )
    print(f"✅ 인덱스 '{INDEX_NAME}' 생성 완료.")
else:
    print(f"✅ 인덱스 '{INDEX_NAME}'가 이미 존재합니다.")

# 3. VectorStore 객체 생성 (업로드 및 검색에 사용)
# 이 객체가 Pinecone 인덱스와 LangChain을 연결합니다.
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)
print(f"✅ PineconeVectorStore 연결 완료.")

⏳ 인덱스 'nist-rag-index'가 존재하지 않아 새로 생성합니다...
✅ 인덱스 'nist-rag-index' 생성 완료.
✅ PineconeVectorStore 연결 완료.


In [25]:
print(f"⏳ 총 {len(chunks)}개의 청크를 Pinecone 인덱스에 업서트합니다...")

# **새 인덱스를 만들고 데이터를 처음 넣는 경우:**
vectorstore = PineconeVectorStore.from_documents(
    chunks, 
    embeddings, 
    index_name=INDEX_NAME, 
    namespace=None # 네임스페이스를 사용하지 않음
)

# **기존 인덱스에 추가하는 경우:**
# vectorstore.add_documents(chunks)

print(f"✅ {len(chunks)}개의 청크 업서트 완료.")

⏳ 총 484개의 청크를 Pinecone 인덱스에 업서트합니다...
✅ 484개의 청크 업서트 완료.


In [26]:
# 1. 기본 검색 테스트 (similarity_search)
query = "AI RMF의 핵심적인 6가지 속성은 무엇인가요?"
results_all = vectorstore.similarity_search(query, k=2)

print("--- 1. 기본 검색 테스트 (필터링 없음) ---")
for doc in results_all:
    print(f"  - [문서]: {doc.metadata.get('doc_title', 'N/A')}")
    print(f"  - [출처]: {doc.metadata.get('source', 'N/A')}, 페이지: {doc.metadata.get('page', 'N/A')}")
    print("-" * 10)
    
# 2. 메타데이터 필터링 검색 테스트 (필수 요구사항)
# source가 'zero_trust'인 문서에서만 검색하도록 필터링 조건을 설정
filter_query = "네트워크 아키텍처에서 Zero Trust의 주요 목표는 무엇입니까?"
zero_trust_filter = {"source": "zero_trust"} 
results_filtered = vectorstore.similarity_search(filter_query, k=3, filter=zero_trust_filter)

print("\n--- 2. 메타데이터 필터링 검색 테스트 (source='zero_trust'만) ---")
if results_filtered:
    for doc in results_filtered:
        print(f"  - [문서]: {doc.metadata.get('doc_title', 'N/A')}")
        print(f"  - [출처]: {doc.metadata.get('source', 'N/A')}, 페이지: {doc.metadata.get('page', 'N/A')}")
        print("-" * 10)
    print("✅ 필터링 검색 성공: 모든 결과가 'zero_trust' 문서에서 나옴.")
else:
    print("❌ 필터링 검색 실패: 결과를 찾을 수 없거나 필터링이 작동하지 않음.")

--- 1. 기본 검색 테스트 (필터링 없음) ---
  - [문서]: nist_ai_risk_framework.pdf
  - [출처]: ai_rmf, 페이지: 46.0
----------
  - [문서]: nist_ai_risk_framework.pdf
  - [출처]: ai_rmf, 페이지: 24.0
----------

--- 2. 메타데이터 필터링 검색 테스트 (source='zero_trust'만) ---
  - [문서]: nist_zero_trust.pdf
  - [출처]: zero_trust, 페이지: 14.0
----------
  - [문서]: nist_zero_trust.pdf
  - [출처]: zero_trust, 페이지: 3.0
----------
  - [문서]: nist_zero_trust.pdf
  - [출처]: zero_trust, 페이지: 10.0
----------
✅ 필터링 검색 성공: 모든 결과가 'zero_trust' 문서에서 나옴.


# 3. RAG Q&A 체인 구현

In [27]:
# --- 라이브러리 임포트 ---
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- LLM 모델 초기화 ---
LLM_MODEL = "gpt-4o-mini" 
llm = ChatOpenAI(
    model=LLM_MODEL, 
    temperature=0.1,
    openai_api_key=os.getenv("OPENAI_API_KEY")
)
print(f"✅ LLM 모델 초기화 완료: {LLM_MODEL}")

✅ LLM 모델 초기화 완료: gpt-4o-mini


In [28]:
# 1. Retriever 정의 (검색 결과 k=4개 사용)
RETRIEVER_K = 4
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_K})

print(f"✅ Retriever 구성 완료 (검색 결과 k={RETRIEVER_K})")

✅ Retriever 구성 완료 (검색 결과 k=4)


In [29]:
# 2. Prompt Template 생성
SYSTEM_TEMPLATE = """
당신은 NIST 문서를 기반으로 답변하는 전문 AI/보안 컨설턴트 보조자입니다.

주어진 '문맥(Context)'만을 사용하여 사용자의 질문에 한국어로 답변하십시오.
만약 주어진 문맥에서 정보를 찾을 수 없다면, '문맥에 기반한 답변을 찾을 수 없습니다.'라고 정중하게 답변하십시오.
답변을 할 때, 반드시 **답변의 근거가 되는 문서 정보**를 답변 마지막에 한 문장으로 요약하여 언급하십시오.

예시:
[답변 내용]
...

[근거]
본 답변은 NIST AI Risk Management Framework (ai_risk)의 'X장' 내용을 참고하였습니다.

---
문맥(Context):
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_TEMPLATE),
        ("human", "질문: {question}"),
    ]
)
print("✅ Prompt Template 구성 완료 (문서 근거 언급 지시 포함)")

✅ Prompt Template 구성 완료 (문서 근거 언급 지시 포함)


In [30]:
# 3. 문서 형식 지정 함수 정의 (Retriever의 출력 형식을 Context로 전달하기 위함)
def format_docs(docs):
    # 문서 내용을 페이지 번호와 제목과 함께 포맷팅
    return "\n\n".join([
        f"--- 문서 제목: {doc.metadata.get('doc_title', 'N/A')} (source: {doc.metadata.get('source', 'N/A')}, page: {doc.metadata.get('page', 'N/A')}) ---\n{doc.page_content}"
        for doc in docs
    ])

# 4. RAG 체인 구축
rag_chain = (
    # 1. 질문을 retriever와 RunnablePassthrough에 전달 (동시 실행)
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    # 2. context와 question을 prompt에 전달
    | prompt
    # 3. prompt를 LLM에 전달
    | llm
    # 4. LLM 출력을 문자열로 파싱
    | StrOutputParser()
)

print("✅ 최종 RAG Runnable Chain 구성 완료")

✅ 최종 RAG Runnable Chain 구성 완료


In [31]:
# LangSmith 설정을 적용한 체인
rag_chain_with_config = rag_chain.with_config(
    {
        "tags": ["nist", "rag", "final_test"], 
        "run_name": "nist_rag_qa_final_run"
    }
)

# 5개 질문 목록
questions = [
    # 1. AI RMF (ai_risk) 관련 질문
    "AI 위험 관리 프레임워크(AI RMF)의 핵심적인 6가지 속성은 무엇이며, 이는 왜 중요한가요?",
    
    # 2. Cybersecurity Framework (cyber_security) 관련 질문
    "CSF 2.0에서 '관리(Govern)' 기능의 주요 활동 목표는 무엇이며, 핵심 결과(KR)에 대해 설명해주세요.",
    
    # 3. Zero Trust (zero_trust) 관련 질문
    "Zero Trust 아키텍처에서 'Never Trust, Always Verify' 원칙이 의미하는 바는 무엇이며, 주요 구성 요소에는 무엇이 있나요?",
    
    # 4. AI RMF 또는 Cybersecurity (혼합) 질문
    "NIST 문서에 따르면, 사이버 보안과 AI 위험 관리 간의 공통적인 연결 고리는 무엇이라고 볼 수 있습니까?",
    
    # 5. Zero Trust (세부 질문)
    "Zero Trust의 논리적 구성 요소 중 정책 엔진(Policy Engine)과 정책 관리자(Policy Administrator)의 역할 차이는 무엇입니까?"
]

print("\n\n=== RAG Q&A 체인 테스트 시작 ===")
for i, q in enumerate(questions):
    print(f"\n[질문 {i+1}] {q}")
    
    # 체인 실행
    response = rag_chain_with_config.invoke(q)
    
    print("-" * 50)
    print(f"[답변 {i+1}]:\n{response}")
    print("=" * 50)

print("=== RAG Q&A 체인 테스트 완료. LangSmith Trace를 확인하세요 ===")



=== RAG Q&A 체인 테스트 시작 ===

[질문 1] AI 위험 관리 프레임워크(AI RMF)의 핵심적인 6가지 속성은 무엇이며, 이는 왜 중요한가요?
--------------------------------------------------
[답변 1]:
AI 위험 관리 프레임워크(AI RMF)의 핵심적인 6가지 속성은 다음과 같습니다:

1. **위험 기반, 자원 효율적, 혁신 촉진, 자발적**: AI RMF는 위험 관리에 중점을 두고 자원을 효율적으로 사용하며 혁신을 촉진하는 방향으로 설계되었습니다.
2. **합의 기반 및 투명한 개발 과정**: 모든 이해관계자가 AI RMF의 개발에 기여할 수 있도록 개방적이고 투명한 과정을 통해 정기적으로 업데이트됩니다.
3. **명확하고 이해하기 쉬운 언어 사용**: 다양한 청중이 이해할 수 있도록 명확한 언어를 사용하며, 기술적 깊이도 충분히 갖추고 있습니다.
4. **공통 언어 및 이해 제공**: AI 위험 관리를 위한 용어, 정의, 메트릭 및 특성을 제공합니다.
5. **사용 용이성 및 다른 위험 관리와의 적합성**: 프레임워크는 직관적으로 사용 가능하며, 조직의 광범위한 위험 관리 전략과 잘 맞아야 합니다.
6. **결과 중심 및 비처방적**: 프레임워크는 특정 행동 지침을 제공하기보다는 결과에 중점을 둡니다.

이러한 속성들은 AI RMF가 다양한 이해관계자와 기술 도메인에 적용 가능하고, AI 시스템의 위험을 효과적으로 관리하는 데 중요한 역할을 합니다.

[근거]
본 답변은 NIST AI Risk Management Framework (ai_risk)의 'Appendix D' 내용을 참고하였습니다.

[질문 2] CSF 2.0에서 '관리(Govern)' 기능의 주요 활동 목표는 무엇이며, 핵심 결과(KR)에 대해 설명해주세요.
--------------------------------------------------
[답변 2]:
'관리(Govern)' 기능의 주요 활동 목표는 조직의 사이버 